# 5 · Evaluation — scoring the arms, the external benchmark, and diagnostics

> **Corrected 13.08.2026** (Selin, review item 11). Until today this file was a stub whose stated
> remit was *"external benchmark and diagnostics"*, held empty because neither source notebook could
> run. Both halves of that have changed. The remit was too narrow — `58fadd7` had already assigned
> this stage the scoring of the training arms — and the blockers it listed are fixed. What that stub
> said is preserved below under [What was believed on 12.08.2026](#what-was-believed-on-12082026),
> because it was true when written.

## What this stage does

Three sections, and they do **not** all run at the same point of the sweep. That is deliberate, but it
means running this notebook top to bottom before R5 will fail halfway — by design, not by breakage.

| § | what it does | runs at | why not earlier |
|---|---|---|---|
| **1 · Score the arms** | the four quantities below, over the out-of-fold predictions `4a_percell_training` writes | **R4** | needs `panel_oof_predictions.csv` and nothing else. It is needed *at* R4 rather than after, because the loss comparison (item 9A) cannot be judged without the calibration slope in §1. |
| **2 · External benchmark** | absorbs [`analysis/evaluation/dreval_benchmark.ipynb`](analysis/evaluation/dreval_benchmark.ipynb) — OncoMLP against DrEval's own baselines, their splits, their metrics | **R5** | consumes retrained outputs and the `auc_cc` targets file, neither of which exists before the sweep runs. |
| **3 · Diagnostics** | absorbs [`analysis/evaluation/diagnostics.ipynb`](analysis/evaluation/diagnostics.ipynb) — proliferation test, input scale, result dispersion | **R5** | same. |

Sections 2 and 3 stay in `analysis/evaluation/` until they run; they move here when they do. Moving a
notebook that raises on its first cell into a numbered stage would put a broken step in the middle of a
chain that is meant to be a path you can walk.

## The four quantities

Fixed by Selin in `58fadd7`, so that the per-cell and MIL architectures are *"comparable by
construction rather than by convention"* — §1 computes them through identical code for both.

| quantity | the question it answers |
|---|---|
| **order** | within a drug, does the predicted ranking of cell lines match the true one? |
| **top-of-order** | are the cell lines called most extreme actually the most extreme? |
| **values** | how far off are the predictions, against a per-drug constant? |
| **spread** | does the model use the real range, or collapse toward the mean? |

**Spread is measured as a calibration slope with its intercept reported alongside** (Selin,
13.08.2026), not as a ratio of standard deviations. The pair is the point: the slope catches
compression, the intercept catches shift, and a model can be flat *and* shifted. Van Calster, Nieboer,
Vickers, Van Calster & Steyerberg, *A calibration hierarchy for risk models*, **J Clin Epidemiol 74
(2016) 167–176**.

## What is not decided yet

These are open and belong to Selin. Each is marked again at the cell that needs it, and no cell
resolves one by being written.

- **The margins for the guards.** The item-9A decision rule is settled in outline: **order is the
  primary quantity** (Selin, 13.08.2026), an arm wins on order, and the other three act as
  non-inferiority guards over **≥3 seeds**. What is *not* settled is what "worse by more than the
  margin" means for each guard. **±0.04 is the seed band on Spearman**, so it is the right bar for
  order and transfers to nothing else: *values* is an error in the target's own units and the
  *calibration slope* is centred on 1.0 on a different scale, so one number would mean three
  different things and two of them would be unsourced. §1 therefore takes a margin **per quantity**,
  each defaulting to that quantity's **own seed band measured from the same ≥3 seeds**, with ±0.04
  recorded as order's prior estimate rather than hard-coded. Selin may replace any of the three with
  a fixed number; that changes three defaults and nothing else.
- **The second baseline** in §1: whether to score against a per-drug constant alone, or also against a
  per-cell-line mean across drugs.
- **Squared or absolute error** for *values*.
- **Folds or seeds** for the dispersion shown in the summary table.

## What was believed on 12.08.2026

The stub this replaces gave two reasons the stage could not be written, and recorded them in a
*"why it cannot run today"* column. Both were accurate then and neither holds now:

- *"`dreval_benchmark` hardcodes `'auc'`, a score `layout.CTRP_SCORES` has rejected since 11.08.2026,
  so `PipelinePaths.build` raises on construction."* — fixed for both notebooks in `e804f07`; they
  read `DEFAULT_CTRP_SCORE`.
- *"It imports the cell-line-effect diagnostic that was deleted on 12.08.2026."* — rewired to DrEval's
  own recipe in `af2cfa9`. That diagnostic stays retired.

Two things the stub flagged **do** still hold, and are not closed by the above:

1. Under leave-cell-line-out, DrEval's normalization removes the **drug** effect only — a held-out
   line's effect is unseen and therefore zero. A synthetic predictor emitting nothing but
   `mean + line effect + drug effect` scores normalized Spearman **0.98**. So *"drug-specific signal,
   or general cell-line fragility?"* is a real question their metric does not answer under our split
   design, and answering it needs a diagnostic that reads held-out labels rather than a metric.
   Whether such a diagnostic returns, and in what form, is still item 11's to settle
   ([`scripts/archive/README.md`](../scripts/archive/README.md)).
2. `dreval_normalize.py` requires a `fold` column so the naive baseline is fitted on the folds a
   prediction did *not* come from. The committed
   `outputs/legacy/panel_void_8drug/panel_oof_predictions.csv` predates that column and the script
   raises on it, correctly. It becomes runnable when [stage 4a](4a_percell_training.ipynb) re-runs.

> ⛔ Nothing is re-run until the 03.08.2026 freeze in [TODO](../docs/TODO.md) lifts.

---

# 1 · Score the arms

**Runs at R4.** Reads one file — `outputs/panel/panel_oof_predictions.csv`, written by
[`4a_percell_training`](4a_percell_training.ipynb) — and writes one, `outputs/panel/panel_metrics.csv`,
plus the summary figure. Nothing here trains anything.

Every row of the input is one **(cell line × drug)** pair, predicted by the fold that held that cell
line out, for one training configuration. The four quantities are computed per drug and then
summarised; the item-9A decision rule is applied in exactly one place, so it can be read rather than
reconstructed.

## 1.1 · Load, and refuse anything the scoring cannot trust

These checks are not ceremony. Each corresponds to a way this table can be wrong while still looking
fine in a spreadsheet, and each would otherwise yield a confident number instead of an error.

The one that matters most is **`ARM_KEYS`** — the columns that together identify one training
configuration. A row must be unique on `(drug, cell_line, *ARM_KEYS)`. If a configuration varies along
a dimension that is *not* in `ARM_KEYS` and not stamped onto the rows — a **seed**, a **loss
function**, a **weighting level** — then several genuinely different predictions collapse onto one key.
Nothing raises, and every later `groupby` silently averages across them. For the seed dimension that is
worse than losing information: the decision rule's margin is *defined* as the spread across seeds, so
pooling them measures that spread as zero and every difference clears every bar.

> ⚠️ **`ARM_KEYS` is provisional and is expected to grow before R4.** Today `4a` stamps only `rep` and
> `weighted`, so that is all a row carries. The item-9A comparison needs at least `seed` and the loss
> arm as well, and `weighted` is a bool that cannot express three weighting levels (off / 0.5 / 1.0).
> Raised 13.08.2026; how the arm is keyed is Selin's decision, and this cell is written so the answer
> is a one-line edit here rather than a rewrite. **Reading an existing file: today's
> `weighted=True` is exactly `alpha=0.5`**, because `density_weighting.DEFAULT_ALPHA` is 0.5 —
> the mapping anyone opening a pre-rename artifact will need, and unrecoverable once it stops
> being obvious. **Until those columns exist, the duplicate check below
> is what stands between a silently-pooled table and a published number.**

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
NB_DIR = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd

OUT = NB_DIR / 'outputs'
OOF_CSV = OUT / 'panel' / 'panel_oof_predictions.csv'
METRICS_CSV = OUT / 'panel' / 'panel_metrics.csv'

#: Columns that together identify one training configuration. PROVISIONAL -- see the markdown above.
#: Extend this the moment 4a stamps `seed` and the loss arm; nothing else in section 1 needs to change.
ARM_KEYS = ['rep', 'weighted']

#: Columns every row must carry regardless of how the arm is keyed. `fold` is required for the same
#: reason dreval_normalize requires it: a prediction that cannot be traced to the split that produced
#: it cannot have an out-of-fold baseline fitted against it.
BASE_COLS = ['drug', 'cell_line', 'fold', 'y_true', 'y_pred']


def load_oof_table(path=OOF_CSV, arm_keys=ARM_KEYS):
    """Read the out-of-fold predictions and refuse the table if scoring it would be misleading.

    Raises rather than warning, in every case. A warning printed above a table of plausible numbers
    is read once and then scrolled past; this notebook exists to decide which loss arm wins, and a
    decision taken on a quietly malformed table is the failure it is meant to prevent.
    """
    if not path.exists():
        raise FileNotFoundError(
            f'{path} does not exist. It is written by 4a_percell_training at R4 of the sweep. '
            f'There is deliberately no fallback to a cached copy: every committed out-of-fold file '
            f'predates the early-stopping fix and the panel rebuild.'
        )
    oof = pd.read_csv(path)

    missing = [c for c in [*BASE_COLS, *arm_keys] if c not in oof.columns]
    if missing:
        raise ValueError(f'{path.name} is missing {missing}. Present: {list(oof.columns)}.')

    nan_rows = oof[['y_true', 'y_pred']].isna().any(axis=1).sum()
    if nan_rows:
        raise ValueError(f'{nan_rows} rows have a null y_true or y_pred; every scored pair needs both.')

    # THE CHECK THAT MATTERS. A duplicate key means some dimension of the run is not stamped on the
    # rows, so distinct predictions are about to be averaged together without anything saying so.
    key = ['drug', 'cell_line', *arm_keys]
    dup = oof.duplicated(subset=key, keep=False)
    if dup.any():
        ex = oof[dup].sort_values(key).head(6)
        raise ValueError(
            f'{int(dup.sum())} rows share a (drug, cell_line, {", ".join(arm_keys)}) key, so more than '
            f'one prediction claims the same slot. Some dimension of the run is not stamped on the '
            f'rows -- seed and the loss arm are the expected culprits. Add it to ARM_KEYS *and* to the '
            f'columns 4a stamps; do not deduplicate. Averaging across seeds here would measure the '
            f'seed band as zero, and the decision rule reads its margin off that band.\n{ex}'
        )

    # One cell line belongs to exactly one fold within an arm, or the split was not grouped by line.
    per_arm_folds = oof.groupby([*arm_keys, 'cell_line'])['fold'].nunique()
    if (per_arm_folds > 1).any():
        bad = per_arm_folds[per_arm_folds > 1]
        raise ValueError(f'{len(bad)} cell lines are held out by more than one fold within an arm:\n{bad.head()}')

    # Every arm must cover the same pairs, or the arms are not being compared on the same data.
    pair_sets = {arm: frozenset(map(tuple, g[['drug', 'cell_line']].to_numpy()))
                 for arm, g in oof.groupby(arm_keys, sort=False)}
    sizes = {a: len(s) for a, s in pair_sets.items()}
    if len(set(pair_sets.values())) > 1:
        raise ValueError(
            f'The arms do not cover identical (drug, cell_line) pairs, so any comparison between them '
            f'is partly a comparison of which pairs they were scored on. Pairs per arm: {sizes}.'
        )

    return oof


oof = load_oof_table()
print(f'{len(oof):,} rows from {OOF_CSV.name}')
print(f'  arms      : {oof.groupby(ARM_KEYS, sort=False).ngroups}  keyed by {ARM_KEYS}')
print(f'  drugs     : {oof.drug.nunique()}')
print(f'  cell lines: {oof.cell_line.nunique()}')
print(f'  folds     : {sorted(oof.fold.unique())}')
print(f'  pairs/arm : {len(oof) // max(oof.groupby(ARM_KEYS, sort=False).ngroups, 1):,}')


## 1.2 · Spread — the calibration slope, with its intercept

Of the four quantities this is the one whose measure is **settled** (Selin, 13.08.2026), so it is
written first; the other three each still carry an open choice and are marked where they land.

The question is whether the model uses the real range of the response or collapses toward the mean.
Regress the **truth on the prediction**, per drug:

$$y_{\text{true}} = a + b \cdot y_{\text{pred}}$$

- **slope $b$** — compression. $b = 1$ means a one-unit change in prediction corresponds to a one-unit
  change in truth. $b > 1$ means the predictions are squeezed toward their own mean and have to be
  stretched to match reality; $b < 1$ means they are over-dispersed.
- **intercept $a$** — shift. Reported alongside, never on its own, because *the pair is the
  instrument*: a model can be flat **and** displaced, and the slope alone cannot tell that from a
  model that is merely flat.

> Van Calster, Nieboer, Vickers, Van Calster & Steyerberg, *A calibration hierarchy for risk models
> was defined: from utopia to empirical data*, **J Clin Epidemiol 74 (2016) 167–176.** Their
> "weak calibration" level is exactly this intercept-and-slope pair.

**Why this rather than the ratio of standard deviations.** A ratio answers only "is the spread right",
and answers it identically for a model that is correctly spread but systematically 0.1 too low. That
matters here specifically: inverse-density weighting is expected to act on *spread*, and the reason its
earlier refutation was never evidence of no effect is that Spearman and MSE are both structurally blind
to spread (`density_weighting.py`, audit 09). Replacing one blind instrument with a half-blind one
would repeat the mistake in a smaller way.

**A note on direction.** Regressing truth on prediction is not interchangeable with regressing
prediction on truth — the slopes are not reciprocals, and only this direction answers "what does a
predicted value tell me about the truth". It is the direction Van Calster et al. specify.

In [ ]:
from scipy.stats import linregress


def calibration(y_true, y_pred, *, min_points=3):
    """Calibration slope and intercept from regressing TRUTH on PREDICTION.

    :returns: ``(slope, intercept)``. Both ``nan`` when the fit is not defined -- fewer than
        ``min_points`` pairs, or a prediction vector with no variance (a model that emitted one
        constant for every cell line of this drug, which is the total-collapse case the slope exists
        to detect). ``nan`` is returned rather than 0 or inf deliberately: a collapsed arm must show
        up as "not measurable here" in the summary, not as a number that averages with real slopes.

    Van Calster et al., J Clin Epidemiol 74 (2016) 167-176 -- their weak-calibration level.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.shape != y_pred.shape:
        raise ValueError(f'shape mismatch: y_true {y_true.shape} vs y_pred {y_pred.shape}')
    if y_true.size < min_points or np.ptp(y_pred) == 0:
        return np.nan, np.nan
    fit = linregress(y_pred, y_true)      # x = prediction, y = truth -- the direction matters
    return float(fit.slope), float(fit.intercept)


## 1.3 · The decision rule — written before the run, evaluated in one place

This is why the notebook is being built at R4 rather than after it. The previous loss comparison
failed because its rule was never written down: it was judged on Spearman and MSE, both of which are
structurally blind to the effect the weighting was supposed to have, and that only became visible
afterwards. A rule fixed after the numbers are seen is not a rule.

**The grid is six arms** — MSE / MAE × `alpha ∈ {0.0, 0.5, 1.0}` on the per-cell architecture, over
**≥3 seeds** ([Step 03](../docs/steps/03-model-and-training-design.md)).

**The rule** (Selin, 13.08.2026): **order is primary**; an arm wins on order, provided it is not worse
than the incumbent on any of the three guards by more than that guard's margin. Non-inferiority on the
guards, superiority on the primary.

### The margin is measured, not chosen

±0.04 is the seed band **on Spearman**, so it is the right bar for order and the wrong bar for
everything else — *values* is an error in the target's own units, and the calibration slope is centred
on 1.0 on a different scale again. One number would mean three different things and two of them would
be unsourced. So each quantity's margin defaults to **its own seed band, measured from the same ≥3
seeds in the same run**. What is pre-registered is the *rule*; the bar being read off the data is the
same shape as fixing a significance level and computing the statistic afterwards.

> ⛔ **`SEED_BAND` is deliberately unset and this section raises until it is filled.** "Seed band" is
> underdetermined: on three seeds, a half-range and a standard deviation differ by roughly a factor of
> two, and that factor *is* the bar. **How the recorded ±0.04 was computed is not written down
> anywhere** — `docs/steps/03` and `docs/TODO.md` both quote it without saying whether it is a range,
> a half-range or an sd — so it cannot be inherited, only re-decided. The blank follows the convention
> `4b_mil_training` uses for `Q2_CONTROL_THRESHOLD`: a pre-registered quantity that must be filled by
> a person, not defaulted by whoever ran the cell.

### Three encodings, stated so they can be rejected

The rule needs to know what "worse" means for each quantity. These are written out rather than left
implicit in comparison operators:

| quantity | better is | rationale |
|---|---|---|
| order | higher | rank correlation |
| top-of-order | higher | more of the true extremes recovered |
| values | lower | it is an error |
| spread | **closer to 1.0** | a calibration slope is not "big is good"; the distance is \|slope − 1\| |

The last is the only one with any choice in it, and it is marked here rather than buried in a
`<` somewhere below.

In [ ]:
#: The quantity that decides. Selin, 13.08.2026.
PRIMARY_QUANTITY = 'order'
GUARD_QUANTITIES = ('top_of_order', 'values', 'spread_slope')

#: What "better" means per quantity -- see the table above. 'toward_one' is the calibration slope.
DIRECTION = {'order': 'higher', 'top_of_order': 'higher',
             'values': 'lower', 'spread_slope': 'toward_one'}

#: Recorded prior estimate of order's seed band (docs/steps/03, docs/TODO.md). NOT a default: the
#: margins are measured from the run. Kept so a measured band wildly unlike it is noticed.
PRIOR_ORDER_BAND = 0.04

#: ⛔ UNSET ON PURPOSE. How a quantity's seed band is computed from its >=3 per-seed values.
#: Fill with a callable, e.g.  lambda v: 0.5 * (np.max(v) - np.min(v))   (half-range)
#:                       or   lambda v: float(np.std(v, ddof=1))          (sample sd)
#: These differ by ~2x on three seeds and that factor IS the decision bar, so it is a person's
#: choice. How the recorded 0.04 was computed is not written down anywhere, so it cannot be inherited.
SEED_BAND = None


def _badness(quantity, value):
    """Map a quantity to a number where LOWER IS ALWAYS WORSE-IS-BIGGER, so one comparison serves all."""
    how = DIRECTION[quantity]
    if how == 'higher':
        return -value
    if how == 'lower':
        return value
    if how == 'toward_one':
        return abs(value - 1.0)
    raise ValueError(f'unknown direction {how!r} for {quantity!r}')


def decide(per_seed, challenger, incumbent, *, seed_band=None):
    """Apply the rule: win on the primary, non-inferior on every guard.

    :param per_seed: {arm: {quantity: [one value per seed]}}. Every arm needs the same quantities
        and at least three seeds; anything less is not a comparison this rule can make.
    :param seed_band: overrides SEED_BAND. Pass a fixed float per quantity via a dict instead of a
        callable if Selin replaces a measured band with a chosen number.
    :returns: dict with the verdict and, importantly, every margin and difference it used -- so the
        decision can be read afterwards rather than recomputed to be believed.
    """
    band = seed_band if seed_band is not None else SEED_BAND
    if band is None:
        raise ValueError(
            'SEED_BAND is unset. It defines the margin every comparison is judged against, and on '
            'three seeds a half-range and an sd differ by about a factor of two. How the recorded '
            '+-0.04 was computed is not documented, so it cannot be inherited -- see the markdown '
            'above. Set SEED_BAND, or pass seed_band=, before any arm is declared a winner.'
        )

    quantities = (PRIMARY_QUANTITY, *GUARD_QUANTITIES)
    for arm in (challenger, incumbent):
        if arm not in per_seed:
            raise KeyError(f'arm {arm!r} not in the per-seed table')
        for q in quantities:
            n = len(per_seed[arm].get(q, ()))
            if n < 3:
                raise ValueError(
                    f'{arm!r} has {n} seed(s) for {q!r}; the rule needs >=3 because the margin is '
                    f'the spread across seeds. With fewer, the bar is undefined, not merely noisy.'
                )

    def margin(q):
        vals = [*per_seed[challenger][q], *per_seed[incumbent][q]]
        return float(band(vals)) if callable(band) else float(band[q])

    report, verdict = {}, True
    for q in quantities:
        c = _badness(q, float(np.mean(per_seed[challenger][q])))
        i = _badness(q, float(np.mean(per_seed[incumbent][q])))
        m = margin(q)
        improvement = i - c                      # positive = challenger is better
        passed = improvement > m if q == PRIMARY_QUANTITY else improvement > -m
        report[q] = {'challenger': c, 'incumbent': i, 'improvement': improvement,
                     'margin': m, 'role': 'primary' if q == PRIMARY_QUANTITY else 'guard',
                     'passed': bool(passed)}
        verdict &= bool(passed)

    return {'challenger': challenger, 'incumbent': incumbent,
            'challenger_wins': bool(verdict), 'per_quantity': report}
